# Path / Labs report pipeline (Databricks)

1. **Automation** – runs `test_script.py` (Playwright + pytest) as a separate process on the driver.
   It logs in every user from `credentials.csv`, pages through the results and saves the reports as PDFs in
   `downloads/<run timestamp>/`.
2. **Extraction** – parses those PDFs **in parallel with PySpark** (PyMuPDF on the executors).
3. **Output** – writes one CSV per PDF, one merged CSV per user and one for the whole run **into the same run folder**
   (so the folder only ever contains PDFs and CSVs), runs data-quality checks, and can save a Delta table.

**Folder layout** (default `base_dir`):
```
/Workspace/Users/shubham.pandey1@procdna.com/HANDSON SANDBOX/
├── test_script.py               ← plain Python file (NOT imported as a notebook)
├── credentials.csv              ← Path, Username, Password
├── downloads/<run timestamp>/   ← PDFs + CSVs of that run only
└── screenshots/                 ← step/failure screenshots, debug html
```
**Requirements:** classic compute, *Dedicated* (single user) access mode, outbound internet (Chromium download + the
target website). Serverless / restricted shared clusters cannot install browser system libraries.
`test_script.py` must be uploaded as a **file** (Workspace → Add → File), not imported as a notebook.

In [0]:
%pip install playwright pymupdf pypdf pytest

Note: you may need to restart the kernel using %restart_python or dbutils.library.restartPython() to use updated packages.


In [0]:
dbutils.library.restartPython()

In [0]:
dbutils.widgets.text("base_dir", "/Workspace/Shared/HANDSON SANDBOX", "Base folder")
dbutils.widgets.dropdown("run_automation", "yes", ["yes", "no"], "Run the browser automation first?")
dbutils.widgets.text("run_id", "", "Run folder (blank = new run / latest existing)")
dbutils.widgets.text("users", "", "Only these usernames (comma separated, blank = all)")
dbutils.widgets.text("limit", "", "Only the first N users (blank = all)")
dbutils.widgets.text("username_pattern", r"user_[A-Za-z0-9]+", "Username regex (in PDF file names)")
dbutils.widgets.text("expected_rows_per_file", "100", "Expected rows per full PDF (0 = skip check)")
dbutils.widgets.text("output_table", "", "Delta table to write (optional, e.g. main.default.pathology_results)")

BASE_DIR = dbutils.widgets.get("base_dir").strip().rstrip("/")
RUN_AUTOMATION = dbutils.widgets.get("run_automation") == "yes"
RUN_ID_PARAM = dbutils.widgets.get("run_id").strip()
USERS = dbutils.widgets.get("users").strip()
LIMIT = dbutils.widgets.get("limit").strip()
USERNAME_PATTERN = dbutils.widgets.get("username_pattern").strip() or r"user_[A-Za-z0-9]+"
EXPECTED_ROWS = int(dbutils.widgets.get("expected_rows_per_file") or 0)
OUTPUT_TABLE = dbutils.widgets.get("output_table").strip()

In [0]:
import os
import subprocess
import sys
import tempfile
from datetime import datetime
BASE_DIR = "/Workspace/Shared/HANDSON SANDBOX"
DOWNLOAD_DIR = os.path.join(BASE_DIR, "downloads")


def _run(cmd):
    r = subprocess.run(cmd, capture_output=True, text=True)
    return r.returncode, (r.stdout + r.stderr)[-1500:]


if RUN_AUTOMATION:
    # --with-deps also installs the Linux libraries Chromium needs (needs root: dedicated classic compute).
    rc, out = _run([sys.executable, "-m", "playwright", "install", "--with-deps", "chromium"])
    if rc != 0:
        print("install --with-deps failed, retrying without system dependencies...\n", out)
        rc, out = _run([sys.executable, "-m", "playwright", "install", "chromium"])
    print("Chromium download:", "OK" if rc == 0 else "FAILED")
    if rc != 0:
        print(out)
        raise RuntimeError(
            "Could not install Chromium. Use a classic, dedicated-access cluster with internet access "
            "(or install the browser with a cluster init script)."
        )

# ---- Shared libraries without root -------------------------------------------------------------
# The notebook user is not root, so `--with-deps` cannot apt-install. Any library that `ldd` reports as
# missing is downloaded with `apt-get download` (user-owned apt dirs) and extracted with `dpkg-deb -x`.
_PW_TMPBASE = tempfile.mkdtemp(prefix="pw_chromium_")
PW_LIB_DIR = os.path.join(_PW_TMPBASE, "pwlibs")
PW_LD_PATH = f"{PW_LIB_DIR}/usr/lib/x86_64-linux-gnu:{PW_LIB_DIR}/lib/x86_64-linux-gnu"
os.environ["LD_LIBRARY_PATH"] = PW_LD_PATH + (":" + os.environ["LD_LIBRARY_PATH"] if os.environ.get("LD_LIBRARY_PATH") else "")

# soname -> Ubuntu 24.04 (noble) package
SONAME_TO_PKG = {
    "libatk-1.0.so.0": "libatk1.0-0t64", "libatk-bridge-2.0.so.0": "libatk-bridge2.0-0t64",
    "libatspi.so.0": "libatspi2.0-0t64", "libXdamage.so.1": "libxdamage1",
    "libXcomposite.so.1": "libxcomposite1", "libXfixes.so.3": "libxfixes3", "libXrandr.so.2": "libxrandr2",
    "libxkbcommon.so.0": "libxkbcommon0", "libgbm.so.1": "libgbm1", "libasound.so.2": "libasound2t64",
    "libcups.so.2": "libcups2t64", "libnss3.so": "libnss3", "libnssutil3.so": "libnss3",
    "libsmime3.so": "libnss3", "libnspr4.so": "libnspr4", "libdrm.so.2": "libdrm2",
    "libpango-1.0.so.0": "libpango-1.0-0", "libcairo.so.2": "libcairo2", "libXext.so.6": "libxext6",
    "libX11.so.6": "libx11-6", "libxcb.so.1": "libxcb1", "libdbus-1.so.3": "libdbus-1-3",
    "libexpat.so.1": "libexpat1", "libgio-2.0.so.0": "libglib2.0-0t64", "libglib-2.0.so.0": "libglib2.0-0t64",
    "libgobject-2.0.so.0": "libglib2.0-0t64", "libudev.so.1": "libudev1", "libXi.so.6": "libxi6",
    "libwayland-server.so.0": "libwayland-server0",
    "libavahi-client.so.3": "libavahi-client3", "libavahi-common.so.3": "libavahi-common3",
}


def chromium_binaries():
    root = os.environ.get("PLAYWRIGHT_BROWSERS_PATH") or os.path.expanduser("~/.cache/ms-playwright")
    pats = ["chromium_headless_shell-*/chrome-headless-shell-linux64/chrome-headless-shell",
            "chromium-*/chrome-linux*/chrome"]
    return sorted(p for pat in pats for p in glob.glob(os.path.join(root, pat)))


def missing_libs():
    missing = set()
    for b in chromium_binaries():
        out = subprocess.run(["ldd", b], capture_output=True, text=True, env=os.environ).stdout
        missing |= {l.split("=>")[0].strip() for l in out.splitlines() if "not found" in l}
    return sorted(missing)


def install_libs_userspace(sonames):
    pkgs = sorted({SONAME_TO_PKG[s] for s in sonames if s in SONAME_TO_PKG})
    unknown = [s for s in sonames if s not in SONAME_TO_PKG]
    if unknown:
        print("No package mapping for:", unknown, "- add them to SONAME_TO_PKG")
    if not pkgs:
        return
    apt, debs = os.path.join(_PW_TMPBASE, "apt"), os.path.join(_PW_TMPBASE, "pwdebs")
    for d in (f"{apt}/lists/partial", f"{apt}/cache/archives/partial", debs, PW_LIB_DIR):
        os.makedirs(d, exist_ok=True)
    opts = ["-o", f"Dir::State::Lists={apt}/lists", "-o", f"Dir::Cache={apt}/cache",
            "-o", "Debug::NoLocking=1", "-o", f"APT::Sandbox::User={getpass.getuser()}"]
    if not os.listdir(f"{apt}/lists") or os.listdir(f"{apt}/lists") == ["partial"]:
        subprocess.run(["apt-get", *opts, "update"], capture_output=True, text=True)
    r = subprocess.run(["apt-get", *opts, "download", *pkgs], cwd=debs, capture_output=True, text=True)
    print("apt-get download", " ".join(pkgs), "->", "OK" if r.returncode == 0 else r.stderr[-800:])
    for deb in glob.glob(os.path.join(debs, "*.deb")):
        subprocess.run(["dpkg-deb", "-x", deb, PW_LIB_DIR], check=True)


LAUNCH_TEST = """
import tempfile, os
from playwright.sync_api import sync_playwright
pdf_path = os.path.join(tempfile.mkdtemp(), 't.pdf')
with sync_playwright() as p:
    b = p.chromium.launch(args=['--no-sandbox', '--disable-dev-shm-usage'])
    pg = b.new_page(); pg.set_content('<h1>ok</h1>'); pg.pdf(path=pdf_path)
    print('Chromium OK', b.version); b.close()
"""

if RUN_AUTOMATION:
    import getpass
    import glob

    for attempt in range(3):              # libraries can pull in further missing libraries
        missing = missing_libs()
        if not missing:
            break
        print("Missing shared libraries:", missing)
        install_libs_userspace(missing)

    # Real launch test in a subprocess (the sync API cannot run inside the notebook event loop).
    r = subprocess.run([sys.executable, "-c", LAUNCH_TEST], capture_output=True, text=True, env=os.environ)
    if r.returncode != 0 or "Chromium OK" not in r.stdout:
        still = missing_libs()
        raise RuntimeError(
            "Chromium was downloaded but cannot start.\n"
            f"ldd 'not found': {still or 'none'}\n\nlaunch stderr:\n{r.stderr[-2000:]}"
        )
    print(r.stdout.strip(), "| LD_LIBRARY_PATH includes", PW_LIB_DIR)
else:
    print("Skipping browser install (run_automation = no)")

install --with-deps failed, retrying without system dependencies...
 Installing dependencies...
Switching to root user to install dependencies...
Failed to install browsers
Error: Installation process exited with code: 1
sudo: a terminal is required to read the password; either use the -S option to read from standard input or configure an askpass helper
sudo: a password is required

Chromium download: OK
Missing shared libraries: ['libXcomposite.so.1', 'libXdamage.so.1', 'libXfixes.so.3', 'libatk-1.0.so.0', 'libatk-bridge-2.0.so.0', 'libatspi.so.0', 'libcups.so.2', 'libxkbcommon.so.0']
apt-get download libatk-bridge2.0-0t64 libatk1.0-0t64 libatspi2.0-0t64 libcups2t64 libxcomposite1 libxdamage1 libxfixes3 libxkbcommon0 -> OK
Missing shared libraries: ['libavahi-client.so.3', 'libavahi-common.so.3']
apt-get download libavahi-client3 libavahi-common3 -> OK
Chromium OK 156.0.8078.4 | LD_LIBRARY_PATH includes /tmp/pw_chromium_c7fwayl0/pwlibs


In [0]:
import subprocess, sys
code = """
from playwright.sync_api import sync_playwright
with sync_playwright() as p:
    b = p.chromium.launch(args=['--no-sandbox','--disable-dev-shm-usage'])
    pg = b.new_page(); pg.set_content('<h1>ok</h1>'); pg.pdf(path='/tmp/t.pdf')
    print('Chromium OK', b.version); b.close()
"""
r = subprocess.run([sys.executable, "-c", code], capture_output=True, text=True, env=os.environ)
print(r.stdout, r.stderr[-2000:])
print("/tmp/t.pdf exists:", os.path.isfile("/tmp/t.pdf"), os.path.getsize("/tmp/t.pdf") if os.path.isfile("/tmp/t.pdf") else "")

Chromium OK 156.0.8078.4
 
/tmp/t.pdf exists: True 4897


In [0]:
def newest_run_dir():
    if not os.path.isdir(DOWNLOAD_DIR):
        return None
    runs = sorted(d for d in os.listdir(DOWNLOAD_DIR) if os.path.isdir(os.path.join(DOWNLOAD_DIR, d)))
    return runs[-1] if runs else None


if RUN_AUTOMATION:
    RUN_ID = RUN_ID_PARAM or datetime.now().strftime("%Y%m%d_%H%M%S")
    script = os.path.join(BASE_DIR, "test_script_cred.py")
    if not os.path.isfile(script):
        raise FileNotFoundError(
            f"{script} not found. Upload test_script.py as a *file* (Workspace -> Add -> File) "
            "into the base folder, together with credentials.csv."
        )
    env = {
        **os.environ,
        "RPA_BASE_DIR": BASE_DIR,
        "RPA_RUN_ID": RUN_ID,
        "PYTHONDONTWRITEBYTECODE": "1",
        "PYTHONUNBUFFERED": "1",
        "LD_LIBRARY_PATH": os.environ.get("LD_LIBRARY_PATH", ""),   # user-space Chromium libs (cell 5)
    }
    if USERS:
        env["RPA_USERS"] = USERS
    if LIMIT:
        env["RPA_LIMIT"] = LIMIT

    # How many users will run (counts only - no credential values are printed).
    # test_script.py logs each user in at URL + their `Path` from credentials.csv.
    import csv
    with open(os.path.join(BASE_DIR, "credentials.csv"), newline="", encoding="utf-8-sig") as fh:
        all_users = [(r.get("Username") or "").strip() for r in csv.DictReader(fh)]
    all_users = [u for u in all_users if u]
    selected = all_users
    if USERS:
        wanted = {u.strip() for u in USERS.split(",") if u.strip()}
        selected = [u for u in selected if u in wanted]
    if LIMIT.isdigit():
        selected = selected[: int(LIMIT)]
    print(f"Users in credentials.csv: {len(all_users)} | selected for this run: {len(selected)}")
    if len(selected) < len(all_users):
        print("  NOTE: the 'users' / 'limit' widgets restrict this run - clear them to run every user.")

    cmd = [sys.executable, "-m", "pytest", "-v", "-s", "-p", "no:cacheprovider", "test_script.py"]
    print("Running:", " ".join(cmd), f"\n  in {BASE_DIR}\n  run folder: downloads/{RUN_ID}\n")
    proc = subprocess.Popen(cmd, cwd=BASE_DIR, env=env, stdout=subprocess.PIPE,
                            stderr=subprocess.STDOUT, text=True, bufsize=1)
    for line in proc.stdout:          # stream the log live
        print(line, end="")
    rc = proc.wait()
    print(f"\npytest exit code: {rc}  (0 = all users passed, 1 = some tests failed)")
else:
    RUN_ID = RUN_ID_PARAM or newest_run_dir()
    if not RUN_ID:
        raise FileNotFoundError(f"No run folders found in {DOWNLOAD_DIR}")

RUN_DIR = os.path.join(DOWNLOAD_DIR, RUN_ID)
print("Run folder:", RUN_DIR)

Users in credentials.csv: 51 | selected for this run: 51
Running: /local_disk0/.ephemeral_nfs/envs/pythonEnv-40656ae6-84f1-474b-a31c-3fd12e450ec5/bin/python -m pytest -v -s -p no:cacheprovider test_script.py 
  in /Workspace/Shared/HANDSON SANDBOX
  run folder: downloads/20261026_110117

============================= test session starts ==============================
platform linux -- Python 3.12.3, pytest-9.1.1, pluggy-1.7.0 -- /local_disk0/.ephemeral_nfs/envs/pythonEnv-40656ae6-84f1-474b-a31c-3fd12e450ec5/bin/python
rootdir: /Workspace/Shared/HANDSON SANDBOX
collecting ... collected 51 items

test_script.py::test_download_path_labs_report[user_a0] [rpa] [user_a0] opening https://musical-biscuit-f1d2b1.netlify.app/a0/
[rpa] [user_a0] clicking 'Sign In'
[rpa] [user_a0] logged in
[rpa] screenshot saved: /Workspace/Shared/HANDSON SANDBOX/screenshots/user_a0_after_login.png
[rpa] [user_a0] clicking 'Path / Labs'
[rpa] clicking 'Path'
[rpa] clicking 'Pending Plan Completion'
[rpa] screensh

In [0]:
import csv
import glob
import re
import shutil
import tempfile

from pyspark.sql import functions as F
from pyspark.sql.types import StringType, StructField, StructType

X_BOUNDS = [
    0, 65, 140, 210, 280, 350, 420,
    480, 545, 635, 660, 700, 755, 792,
]

COLUMN_NAMES = [
    "result_date",
    "patient_raw",
    "facility",
    "location",
    "result",
    "actions_and_plans",
    "comments_assigned_to",
    "flags",
    "notifications",
    "notes",
    "patient_portal",
    "results_processed",
    "photos",
]

RECORD_COLUMNS = [
    "result_date", "patient_name", "dob", "mrn", "facility", "location", "result",
    "actions_and_plans", "comments_assigned_to", "flags", "notifications", "notes",
    "patient_portal", "results_processed", "photos", "source_page",
]
FILE_COLUMNS = ["source_file", "username", "downloaded_at", "report_page", "record_seq"]
ALL_COLUMNS = RECORD_COLUMNS + FILE_COLUMNS

# Match the validated extraction notebook's table-body bounds.
# These bounds exclude PDF header/footer content and prevent those words from
# being attached to the first/last pathology records.
BODY_Y_MIN = 65
BODY_Y_MAX = 570

DATE_PATTERN = r"^\d{2}/\d{2}/\d{4}$"
DOB_PATTERN = r"\bDOB\s+(\d{2}/\d{2}/\d{4})\b"
MRN_PATTERN = r"\bMRN\s+([A-Z]{2,})\s*-\s*(\d+)\b"


def clean_text(value):
    """Normalize whitespace without removing meaningful punctuation."""
    if not value:
        return None
    value = re.sub(r"\s+", " ", value).strip()
    return value or None


def unique_join(values, separator=" "):
    """Join multiline values while removing consecutive duplicates."""
    cleaned = [clean_text(v) for v in values]
    cleaned = [v for v in cleaned if v]
    output = []
    for value in cleaned:
        if not output or output[-1] != value:
            output.append(value)
    return separator.join(output) if output else None


def words_to_lines(words, tolerance=2.5):
    """Group PDF words by approximate vertical position."""
    lines = []
    for word in sorted(words, key=lambda w: (w[1], w[0])):
        x0, y0, x1, y1, text = word[:5]
        target = None
        for line in lines:
            if abs(line["y"] - y0) <= tolerance:
                target = line
                break
        if target is None:
            target = {"y": y0, "words": []}
            lines.append(target)
        target["words"].append(word)
    for line in lines:
        line["words"].sort(key=lambda w: w[0])
    return sorted(lines, key=lambda line: line["y"])


def extract_column_text(words, x_left, x_right):
    """Extract words that fall inside a specified column."""
    selected = [w for w in words if x_left <= w[0] < x_right]
    lines = words_to_lines(selected)
    return unique_join([" ".join(word[4] for word in line["words"]) for line in lines])


def extract_pdf_records(pdf_path):
    """Extract records using the validated PDF-to-CSV notebook's layout logic."""
    import pymupdf  # imported here so it resolves on every worker

    date_re = re.compile(DATE_PATTERN)
    dob_re = re.compile(DOB_PATTERN)
    mrn_re = re.compile(MRN_PATTERN)
    records = []

    with pymupdf.open(str(pdf_path)) as pdf:
        for page_number, page in enumerate(pdf, start=1):
            # Match the known-accurate extractor's table-body band.
            words = [
                w for w in page.get_text("words", sort=True)
                if BODY_Y_MIN <= w[1] < BODY_Y_MAX
            ]

            # Every record starts with a date in the first column.
            anchors = []
            for word in words:
                x0, y0, x1, y1, text = word[:5]
                if x0 < X_BOUNDS[1] and date_re.fullmatch(text.strip()):
                    anchors.append((y0, text.strip()))
            anchors.sort(key=lambda item: item[0])

            for index, (start_y, date_text) in enumerate(anchors):
                # Stop at the next record's date; last record ends at body boundary.
                end_y = (
                    anchors[index + 1][0]
                    if index + 1 < len(anchors)
                    else BODY_Y_MAX
                )
                record_words = [
                    w for w in words if start_y - 1 <= w[1] < end_y
                ]

                row = {}
                for i, column in enumerate(COLUMN_NAMES):
                    row[column] = extract_column_text(
                        record_words, X_BOUNDS[i], X_BOUNDS[i + 1]
                    )

                row["result_date"] = date_text
                row["source_page"] = str(page_number)

                patient_raw = row.get("patient_raw") or ""

                # DOB/MRN can be split by PDF layout or have spaces around the
                # MRN hyphen (for example, "MRN NF- 180060"). Parse the
                # identifiers independently and never leave their labels in the name.
                dob_match = dob_re.search(patient_raw)
                mrn_match = mrn_re.search(patient_raw)
                row["dob"] = dob_match.group(1) if dob_match else None
                row["mrn"] = (
                    f"{mrn_match.group(1)}-{mrn_match.group(2)}"
                    if mrn_match else None
                )

                # Patient name is only the text before the DOB line. In some
                # browser-generated PDFs, adjacent Facility text leaks into the
                # Patient column; cut it off at the facility anchor before cleaning.
                patient_name = patient_raw
                if dob_match:
                    patient_name = patient_name[:dob_match.start()]
                else:
                    mrn_pos = re.search(r"\bMRN\b", patient_name, re.IGNORECASE)
                    if mrn_pos:
                        patient_name = patient_name[:mrn_pos.start()]

                # Defensive cleanup for adjacent-column spillover seen in these reports.
                patient_name = re.split(
                    r"\bNorthfield\b|\bDermatology\b|\bSurgery\s+Center\b",
                    patient_name,
                    maxsplit=1,
                    flags=re.IGNORECASE,
                )[0]
                patient_name = re.sub(r"\bDOB\b.*$", "", patient_name, flags=re.IGNORECASE)
                row["patient_name"] = clean_text(patient_name)
                records.append(row)

    return records


def parse_file_name(path):
    """<name>_<username>_<YYYYMMDD_HHMMSS>[_pN].pdf -> (username, 'YYYYMMDD_HHMMSS', page)."""
    stem = os.path.splitext(os.path.basename(path))[0]
    m = re.search(
        rf"(?P<user>{USERNAME_PATTERN})_(?P<ts>\d{{8}}_\d{{6}})(?:_p(?P<page>\d+))?$", stem
    )
    if not m:
        return None, None, "1"
    return m.group("user"), m.group("ts"), m.group("page") or "1"


def extract_file(path):
    """
    Executor task: returns (path, error_or_None, [row tuples in ALL_COLUMNS order]).
    A corrupt PDF is reported instead of failing the whole job.
    """
    try:
        recs = extract_pdf_records(path)
    except Exception as exc:  # noqa: BLE001
        return (path, f"{type(exc).__name__}: {exc}", [])
    user, ts, page_no = parse_file_name(path)
    fname = os.path.basename(path)
    rows = [
        tuple(rec.get(c) for c in RECORD_COLUMNS) + (fname, user, ts, page_no, str(seq))
        for seq, rec in enumerate(recs, start=1)
    ]
    return (path, None, rows)

In [0]:

pdf_files = sorted(glob.glob(os.path.join(RUN_DIR, "*.pdf")))
print(f"{len(pdf_files)} PDF(s) found in {RUN_DIR}")
if not pdf_files:
    raise FileNotFoundError(f"No PDFs in {RUN_DIR} - check the automation log above.")

153 PDF(s) found in /Workspace/Shared/HANDSON SANDBOX/downloads/20261026_110117


In [0]:

schema = StructType([StructField(c, StringType(), True) for c in ALL_COLUMNS])

results = [extract_file(f) for f in pdf_files]

errors = [(path, err) for path, err, _ in results if err is not None]
for path, err in errors:
    print(f"ERROR parsing {os.path.basename(path)}: {err}")

all_rows = [row for _, _, rows in results for row in rows]
if not all_rows:
    raise ValueError(
        "No pathology records were extracted. Check the PDF layout, date anchors, "
        "and BODY_Y_MIN/BODY_Y_MAX before writing CSV files."
    )

raw_df = spark.createDataFrame(all_rows, schema)

pathology_df = (
    raw_df
    .withColumn("result_date", F.to_date("result_date", "MM/dd/yyyy"))
    .withColumn("dob", F.to_date("dob", "MM/dd/yyyy"))
    .withColumn("downloaded_at", F.to_timestamp("downloaded_at", "yyyyMMdd_HHmmss"))
    .withColumn("report_page", F.col("report_page").cast("int"))
    .withColumn("source_page", F.col("source_page").cast("int"))
    .withColumn("record_seq", F.col("record_seq").cast("int"))
    .cache()
)

display(pathology_df.limit(10))

2026-10-26 11:15:45.499292770 [W:onnxruntime:Default, device_discovery.cc:95 GetPciBusId] Skipping pci_bus_id for PCI path at "/sys/devices/LNXSYSTM:00/LNXSYBUS:00/ACPI0004:00/MSFT1000:00/5620e0c7-8062-4dce-aeb7-520c7ef76171" because filename "5620e0c7-8062-4dce-aeb7-520c7ef76171" did not match expected pattern of [0-9a-f]+:[0-9a-f]+:[0-9a-f]+[.][0-9a-f]+


result_date,patient_name,dob,mrn,facility,location,result,actions_and_plans,comments_assigned_to,flags,notifications,notes,patient_portal,results_processed,photos,source_page,source_file,username,downloaded_at,report_page,record_seq
2026-10-12,"Chen, Kenji I.",null,NF-180060,A: right & Surgery,"shin Seborrheic Keratosis,","Reassure: irritated benign, further treatment",no,BENIGN,"Free Form Okafor, 10/14/26 PM",- Priya - 03:30,Release,10/13/2026 03:42 FINAL,PM,null,1,Northfield_EMR_user_a0_20261026_110129_p1.pdf,user_a0,2026-10-26T11:01:29Z,1,1
2026-10-12,"Donovan,",null,NF-165073,A: mid & back Surgery,upper Squamous Carcinoma,Cell Schedule: in situ visit to recheck and discuss options,SRT VS EXC VS MOHS site,CANCEROUS,"Free Form Nguyen, 10/15/26 PM",- Tara - 04:24,Hide,10/13/2026 04:50 FINAL,PM,null,1,Northfield_EMR_user_a0_20261026_110129_p1.pdf,user_a0,2026-10-26T11:01:29Z,1,2
2026-10-11,"Underwood,",null,NF-157746,A: right & ala Surgery,"nasal Basal Cell Carcinoma, type",Schedule: nodular Mohs consult,MOHS - Dr. surgery Lindqvist,CANCEROUS,"Free Form Jordan 03:32 Free Form Brooks, 10/18/26 AM","- Hale, - 10/16/26 PM - Samir - 09:46",Hide,10/14/2026 05:08 FINAL,PM,null,1,Northfield_EMR_user_a0_20261026_110129_p1.pdf,user_a0,2026-10-26T11:01:29Z,1,3
2026-10-10,"Marchetti,",null,NF-235852,A: right & Surgery,shin Atypical Intraepidermal Melanocytic Proliferation,Schedule: excision,re- RE-EXC - Hale,ATYPICAL,Free Form Jordan 02:18,"- Hale, Call daughter - 10/12/26 (HIPAA on file) PM",Hide,10/11/2026 08:15 FINAL,AM,null,1,Northfield_EMR_user_a0_20261026_110129_p1.pdf,user_a0,2026-10-26T11:01:29Z,1,4
2026-10-09,"Rasmussen,",null,NF-202803,A: left & preauricular Surgery cheek,"Basal Cell Carcinoma, superficial",Schedule: ED&C type topical,ED&C VS 5- vs FU 5-FU,CANCEROUS,"Free Form Ramos, 10/15/26 PM",- Elena - 02:29,Hide,10/12/2026 05:04 FINAL,PM,null,1,Northfield_EMR_user_a0_20261026_110129_p1.pdf,user_a0,2026-10-26T11:01:29Z,1,5
2026-10-08,"Sandoval,",null,NF-249385,A: right & arm Surgery,upper Epidermal Cyst,Inclusion Reassure: benign,null,BENIGN,"Free Form Okafor, 10/11/26 PM",- Priya - 04:59,Release Powered,10/10/2026 03:24 FINAL by,PM Netlify,null,1,Northfield_EMR_user_a0_20261026_110129_p1.pdf,user_a0,2026-10-26T11:01:29Z,1,6
2026-10-08,"Fitzgerald,",null,NF-222142,A: left & central Surgery cheek,inferior Atypical malar Intraepidermal Melanocytic Proliferation,Schedule: excision,re- RE-EXC - Hale,ATYPICAL,Free Form Jordan 10:17,"- Hale, - 10/12/26 AM",Hide,10/11/2026 05:05 FINAL,PM,null,2,Northfield_EMR_user_a0_20261026_110129_p1.pdf,user_a0,2026-10-26T11:01:29Z,1,7
2026-10-08,"Abernathy,",null,NF-204402,A: right & index Surgery,"dorsal Basal Cell finger Carcinoma, type",Schedule: nodular Mohs consult,MOHS - Dr. surgery Lindqvist,CANCEROUS,"Free Form Nguyen, 10/12/26 PM Free Form Ramos, 10/14/26 AM",- Tara - 03:59 - Elena - 08:21,Hide,10/10/2026 04:57 FINAL,PM,null,2,Northfield_EMR_user_a0_20261026_110129_p1.pdf,user_a0,2026-10-26T11:01:29Z,1,8
2026-10-07,"Sandoval,",null,NF-208058,A: chin & Surgery,Hypertrophic Keratosis,Actinic Reassure: treat with LN2 at visit,LN2 NEXT VISIT next,PRECANCEROUS,"Free Form Ramos, 10/09/26 AM",- Elena - 08:22,Release,10/08/2026 11:50 FINAL,AM,null,2,Northfield_EMR_user_a0_20261026_110129_p1.pdf,user_a0,2026-10-26T11:01:29Z,1,9
2026-10-05,"Sandoval,",null,NF-158748,A: left & posterior Surgery,"Basal Cell neck Carcinoma, superficial",Schedule: ED&C type topical,ED&C VS 5- vs FU 5-FU,CANCEROUS,"Free Form Brooks, 10/11/26 PM",- Samir - 01:10,Hide,10/08/2026 08:34 FINAL,AM,null,2,Northfield_EMR_user_a0_20261026_110129_p1.pdf,user_a0,2026-10-26T11:01:29Z,1,10


In [0]:
# No CSV files are written. Per-user combined ('_all') data is saved as Delta tables only.
TABLE_SCHEMA = "us_scdb_phi_dev.comm_bronze"   # catalog.schema for the per-user combined tables

user_files = (
    pathology_df.groupBy("username")
    .agg(F.sort_array(F.collect_set("source_file")).alias("files"))
    .collect()
)

tables_written = []
for r in user_files:                                      # one table per user (only if several pages)
    user, files = r["username"], r["files"]
    if len(files) > 1:
        first_stem = re.sub(r"_p\d+$", "", os.path.splitext(files[0])[0])
        name = first_stem + "_all.csv"                    # name the combined file would have had
        table_name = f"{TABLE_SCHEMA}.`{name[15:-24]}`"
        (pathology_df.filter(F.col("username") == user)
            .orderBy("report_page", "record_seq")
            .write.mode("overwrite").option("overwriteSchema", "true")
            .saveAsTable(table_name))
        tables_written.append(table_name)

print(f"{len(tables_written)} combined table(s) written:", tables_written)

154 CSV file(s) written to /Workspace/Shared/HANDSON SANDBOX/downloads/20261026_110117
51 combined table(s) written: ['us_scdb_phi_dev.comm_bronze.`user_a0_`', 'us_scdb_phi_dev.comm_bronze.`user_a10_`', 'us_scdb_phi_dev.comm_bronze.`user_a11_`', 'us_scdb_phi_dev.comm_bronze.`user_a12_`', 'us_scdb_phi_dev.comm_bronze.`user_a13_`', 'us_scdb_phi_dev.comm_bronze.`user_a14_`', 'us_scdb_phi_dev.comm_bronze.`user_a15_`', 'us_scdb_phi_dev.comm_bronze.`user_a16_`', 'us_scdb_phi_dev.comm_bronze.`user_a17_`', 'us_scdb_phi_dev.comm_bronze.`user_a18_`']
['Northfield_EMR_user_a0_20261026_110129_p1.csv', 'Northfield_EMR_user_a0_20261026_110129_p1.pdf', 'Northfield_EMR_user_a0_20261026_110138_p2.csv', 'Northfield_EMR_user_a0_20261026_110138_p2.pdf', 'Northfield_EMR_user_a0_20261026_110142_p3.csv', 'Northfield_EMR_user_a0_20261026_110142_p3.pdf', 'Northfield_EMR_user_a10_20261026_110419_p1.csv', 'Northfield_EMR_user_a10_20261026_110419_p1.pdf', 'Northfield_EMR_user_a10_20261026_110425_p2.csv', 'Northfi

In [0]:
total_rows = pathology_df.count()
print("Extracted rows:", total_rows)

rows_per_file = (
    pathology_df.groupBy("source_file", "username", "report_page").count()
    .orderBy("username", "report_page", "source_file")
)
display(rows_per_file)
if EXPECTED_ROWS > 0:
    short = rows_per_file.filter(F.col("count") < EXPECTED_ROWS)
    print(f"Files with fewer than {EXPECTED_ROWS} rows (the last page of a user can legitimately be shorter):",
          short.count())
    display(short)

print("Rows with missing patient_name or mrn:")
display(
    pathology_df.filter(F.col("patient_name").isNull() | F.col("mrn").isNull())
    .select("source_file", "result_date", "patient_name", "dob", "mrn", "source_page")
)

print("Duplicate MRNs per user:")
display(
    pathology_df.groupBy("username", "mrn").count()
    .filter(F.col("count") > 1)
    .orderBy(F.desc("count"))
)

display(
    pathology_df.select(
        "source_file", "result_date", "patient_name", "dob", "mrn", "location", "result",
        "actions_and_plans", "comments_assigned_to", "flags", "notifications",
        "notes", "patient_portal", "results_processed",
    )
    .orderBy(F.col("result_date").desc())
    .limit(25)
)

required_columns = ["result_date", "patient_name", "mrn", "result", "flags"]
display(
    pathology_df.select([
        F.sum(
            F.when(F.col(c).isNull() | (F.trim(F.col(c).cast("string")) == ""), 1).otherwise(0)
        ).alias(f"{c}_missing")
        for c in required_columns
    ])
)

Extracted rows: 12750


source_file,username,report_page,count
Northfield_EMR_user_a0_20261026_110129_p1.pdf,user_a0,1,100
Northfield_EMR_user_a0_20261026_110138_p2.pdf,user_a0,2,100
Northfield_EMR_user_a0_20261026_110142_p3.pdf,user_a0,3,50
Northfield_EMR_user_a1_20261026_110149_p1.pdf,user_a1,1,100
Northfield_EMR_user_a1_20261026_110155_p2.pdf,user_a1,2,100
Northfield_EMR_user_a1_20261026_110200_p3.pdf,user_a1,3,50
Northfield_EMR_user_a10_20261026_110419_p1.pdf,user_a10,1,100
Northfield_EMR_user_a10_20261026_110425_p2.pdf,user_a10,2,100
Northfield_EMR_user_a10_20261026_110430_p3.pdf,user_a10,3,50
Northfield_EMR_user_a11_20261026_110435_p1.pdf,user_a11,1,100


Files with fewer than 100 rows (the last page of a user can legitimately be shorter): 51


source_file,username,report_page,count
Northfield_EMR_user_a0_20261026_110142_p3.pdf,user_a0,3,50
Northfield_EMR_user_a1_20261026_110200_p3.pdf,user_a1,3,50
Northfield_EMR_user_a10_20261026_110430_p3.pdf,user_a10,3,50
Northfield_EMR_user_a11_20261026_110446_p3.pdf,user_a11,3,50
Northfield_EMR_user_a12_20261026_110503_p3.pdf,user_a12,3,50
Northfield_EMR_user_a13_20261026_110520_p3.pdf,user_a13,3,50
Northfield_EMR_user_a14_20261026_110537_p3.pdf,user_a14,3,50
Northfield_EMR_user_a15_20261026_110553_p3.pdf,user_a15,3,50
Northfield_EMR_user_a16_20261026_110609_p3.pdf,user_a16,3,50
Northfield_EMR_user_a17_20261026_110626_p3.pdf,user_a17,3,50


Rows with missing patient_name or mrn:


source_file,result_date,patient_name,dob,mrn,source_page


Duplicate MRNs per user:


username,mrn,count


source_file,result_date,patient_name,dob,mrn,location,result,actions_and_plans,comments_assigned_to,flags,notifications,notes,patient_portal,results_processed
Northfield_EMR_user_a34_20261026_111057_p1.pdf,2026-10-12,"Kaminski,",null,NF-159059,dorsal Dysplastic with mild margins,"Nevus Reassure: atypia, margins clear",clear,ATYPICAL,"Free Form Okafor, 10/15/26 PM",- Priya - 01:34,Hide,10/13/2026 08:39 FINAL,AM
Northfield_EMR_user_a49_20261026_111510_p1.pdf,2026-10-12,"Jablonski,",null,NF-193386,lip Squamous Carcinoma,Cell Schedule: in situ visit to recheck and discuss options,SRT VS EXC VS MOHS site,CANCEROUS,"Free Form Nguyen, 10/17/26 AM Free Form Brooks, 10/20/26 PM",- Tara - 09:21 - Samir - 12:34,Hide,10/14/2026 01:24 FINAL,PM
Northfield_EMR_user_a13_20261026_110509_p1.pdf,2026-10-12,"Okonkwo,",null,NF-212049,"left Basal Cell Carcinoma, type",Schedule: nodular Mohs consult,MOHS - surgery Lindqvist,Dr. CANCEROUS,"Free Brooks, 10/16/26 PM",Form - Samir - 03:29,Hide,10/14/2026 03:32 FINAL,PM
Northfield_EMR_user_a45_20261026_111401_p1.pdf,2026-10-12,"Moreau,",null,NF-188756,right upper Hypertrophic Keratosis,Actinic Reassure: treat at next,LN2 NEXT with LN2 VISIT visit,PRECANCEROUS,"Free Ramos, 10/17/26 AM",Form - Elena - 08:22,Release,10/14/2026 08:49 FINAL,AM
Northfield_EMR_user_a25_20261026_110828_p1.pdf,2026-10-12,"Toomey,",null,NF-160832,helical Squamous Carcinoma,Cell Schedule: in situ visit to recheck and discuss options,SRT VS EXC VS MOHS site,CANCEROUS,"Free Form Brooks, 10/17/26 PM",- Samir - 01:40,Hide,10/14/2026 12:29 FINAL,PM
Northfield_EMR_user_a45_20261026_111401_p1.pdf,2026-10-12,"Harrington,",null,NF-175267,"mid Squamous Carcinoma, keratoacanthoma type",Cell Schedule: Mohs,MOHS - surgery Lindqvist,Dr. CANCEROUS,"Free Nguyen, 10/14/26 AM",Form - Tara - 09:46,Hide,10/13/2026 03:29 FINAL,PM
Northfield_EMR_user_a10_20261026_110419_p1.pdf,2026-10-12,"Forsythe,",null,NF-168973,tip Actinic Keratosis,Reassure: field therapy with topical 5-FU,5-FU FIELD TX,PRECANCEROUS,"Free Form Brooks, 10/18/26 PM",- Hx of organ Samir - transplant 01:38,Release,10/15/2026 05:38 FINAL,PM
Northfield_EMR_user_a45_20261026_111401_p1.pdf,2026-10-12,"Donovan,",null,NF-172623,left Lichenoid neck,"Keratosis Reassure: benign, further treatment",no,BENIGN,"Free Brooks, 10/16/26 AM Free Ramos, 10/17/26 PM",Form - Samir - 09:41 Form - Elena - 03:30,Release,10/14/2026 11:25 FINAL,AM
Northfield_EMR_user_a33_20261026_111040_p1.pdf,2026-10-12,"Abernathy,",null,NF-245949,tip No Residual Squamous Carcinoma,Reassure: Cell margins in situ,clear,CANCEROUS,Free Form Jordan 04:15,"- Hale, - 10/16/26 PM",Hide,10/14/2026 09:18 FINAL,AM
Northfield_EMR_user_a45_20261026_111401_p1.pdf,2026-10-12,"Kaminski,",null,NF-203368,left earlobe Lichenoid,"Keratosis Reassure: benign, further treatment",no,BENIGN,"Free Ramos, 10/17/26 PM",Form - Elena - 12:30,Release,10/15/2026 11:40 FINAL,AM


result_date_missing,patient_name_missing,mrn_missing,result_missing,flags_missing
0,0,0,0,0


In [0]:
pathology_df.write.mode("overwrite").option("overwriteSchema", "true").saveAsTable('us_scdb_phi_dev.comm_bronze.output')

In [0]:
# Remove this run's download folder and everything in it, now that the data is saved to tables.
run_dir_real = os.path.realpath(RUN_DIR)
download_dir_real = os.path.realpath(DOWNLOAD_DIR)
# Safety check: only delete a subfolder of DOWNLOAD_DIR, never DOWNLOAD_DIR itself or anything outside it.
if os.path.dirname(run_dir_real) != download_dir_real:
    raise ValueError(f"Refusing to delete {RUN_DIR}: not a run folder directly under {DOWNLOAD_DIR}")

if os.path.isdir(run_dir_real):
    shutil.rmtree(run_dir_real)
print("Deleted run folder:", RUN_DIR, "| still exists:", os.path.exists(run_dir_real))